# Activity 2: Simple House Price Prediction
**Machine Learning CIE 2 | Unit II – Supervised Learning: Regression**

## Aim
Build a Linear Regression model to predict a continuous house-value target, evaluate it on unseen test data, visualise results and make one new prediction.

## Dataset
California Housing dataset from scikit-learn, originally obtained from StatLib. It contains 20,640 census-block-group observations, eight numerical features and the continuous target `MedHouseVal`. The target is expressed in units of $100,000. This is historical district-level data, not current individual-house listing data.

**Source:** https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset

> Run all cells in order. The first run downloads the dataset and therefore requires internet access. Metrics and plots are generated by the code; no results are fabricated.

## 1. Import libraries and load the dataset

In [ ]:
# If needed in Google Colab, uncomment:
# !pip install -r requirements.txt

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

ROOT = Path.cwd()
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
print("Dataset shape:", df.shape)
display(df.head())
display(df.tail())

## 2. Inspect columns, data types, statistics and data quality

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types:")
display(df.dtypes)
print("\nSummary statistics:")
display(df.describe().round(3))
print("\nMissing values per column:")
display(df.isna().sum())
print("Duplicate rows:", int(df.duplicated().sum()))

The dataset contains eight numerical input features and one continuous target. The target is `MedHouseVal`. The dataset documentation reports no missing values. The code checks missing values and duplicates directly. No record ID is used as a predictor.

## 3. Select input feature (X) and target (y)

In [ ]:
feature = "MedInc"       # median income in the block group
target = "MedHouseVal"   # median house value, in $100,000 units
X = df[[feature]]
y = df[target]

print("Input feature:", feature)
print("Target:", target)
print("X shape:", X.shape, "| y shape:", y.shape)

`MedInc` is selected because income can be associated with housing values. A single feature is used to make the simple linear regression coefficient and equation easy to interpret. Association does not prove causation.

## 4. Visualise the relationship

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X[feature], y, alpha=0.25, s=10)
plt.title("Median Income vs Median House Value")
plt.xlabel("Median income (tens of thousands of USD)")
plt.ylabel("Median house value (units of $100,000)")
plt.tight_layout()
plt.savefig(FIG / "income_vs_house_value.png", dpi=160)
plt.show()

## 5. Split into training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

The 80:20 split uses the training set to learn model parameters and reserves the test set for evaluation on unseen observations. The test set is not used to train the model.

## 6. Train Linear Regression and record equation

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
coefficient = float(model.coef_[0])
intercept = float(model.intercept_)
print("Intercept:", intercept)
print("Coefficient:", coefficient)
print(f"Model equation: MedHouseVal = {intercept:.6f} + ({coefficient:.6f} × MedInc)")

The coefficient (slope) estimates the change in predicted median house value, in $100,000 units, for a one-unit increase in median income. The intercept is the model's estimated target when median income is zero.

## 7. Generate test-set predictions

In [ ]:
predictions = model.predict(X_test)
comparison = pd.DataFrame({
    "Actual_MedHouseVal": y_test.reset_index(drop=True),
    "Predicted_MedHouseVal": pd.Series(predictions)
})
display(comparison.head(15).round(3))
comparison.head(20).to_csv(OUT / "actual_vs_predicted_sample.csv", index=False)

## 8. Evaluate using MAE, MSE, RMSE and R²

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)

metrics = {
    "MAE": float(mae), "MSE": float(mse), "RMSE": float(rmse), "R2": float(r2),
    "train_rows": int(len(X_train)), "test_rows": int(len(X_test)),
    "coefficient": coefficient, "intercept": intercept
}
display(pd.DataFrame([metrics]).T.rename(columns={0: "Value"}))
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))

**Interpretation:** MAE is the average absolute error; MSE is the average squared error; RMSE is in the target's units; R² compares the model with a mean-based baseline and can be negative on test data. Because `MedHouseVal` is in units of $100,000, MAE and RMSE can be converted to approximate USD by multiplying by 100,000.

## 9. Plot actual vs predicted values

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions, alpha=0.25, s=10)
low = min(float(y_test.min()), float(predictions.min()))
high = max(float(y_test.max()), float(predictions.max()))
plt.plot([low, high], [low, high], linestyle="--", label="Perfect prediction")
plt.title("Actual vs Predicted Median House Values")
plt.xlabel("Actual median house value ($100,000 units)")
plt.ylabel("Predicted median house value ($100,000 units)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG / "actual_vs_predicted.png", dpi=160)
plt.show()

Points closer to the dashed reference line indicate more accurate predictions. Points far from the line indicate larger prediction errors.

## 10. Predict a new example

In [ ]:
# Choose a reasonable input: median MedInc from the training data.
new_income = float(X_train[feature].median())
new_prediction = float(model.predict(pd.DataFrame({feature: [new_income]}))[0])
new_result = {
    "MedInc": new_income,
    "Predicted_MedHouseVal_100k_USD": new_prediction,
    "Predicted_USD_approx": new_prediction * 100000
}
print(new_result)
(OUT / "new_prediction.json").write_text(json.dumps(new_result, indent=2))

## 11. Save model and final outputs

In [ ]:
df.to_csv(OUT / "california_housing_loaded.csv", index=False)
joblib.dump(model, OUT / "linear_regression_model.joblib")
print("Saved files:")
for path in sorted(list(OUT.glob("*")) + list(FIG.glob("*"))):
    print(path)

## 12. Conclusion and limitations

A simple Linear Regression model was trained to estimate California district-level median house value using median income as one input feature. The model was evaluated on a held-out test set using MAE, MSE, RMSE and R², and the actual-versus-predicted plot was inspected.

This one-feature model is easy to interpret but cannot represent all factors affecting house prices. The data is aggregated at census-block-group level and comes from the 1990 census; it should not be used to estimate current prices of individual homes. Run the notebook and use its displayed metric values in your final submission.

### Faculty acknowledgement
This project was completed as part of the Machine Learning CIE 2 activity. Faculty reference and Machine Learning learning resources: https://github.com/Princy2129